## 🎯 Learning Objectives
* Understand the concept and importance of streaming output in LangGraph.
* Differentiate between `invoke()` and `stream()` methods for graph execution.
* Implement and interpret streamed output from a LangGraph agent.
* Identify common use cases and performance considerations for streaming in agentic workflows.


## Streaming Graph Execution Output: Real-time Insights into Your Agent's Mind

In the world of AI agents, responsiveness and transparency are paramount. Imagine asking a complex question to an AI assistant, and instead of waiting in silence for a minute, you see its thought process unfold in real-time: "*Okay, I need to check the weather. Searching for 'weather in London'... Found current conditions... Now, I'll summarize this for you.*" This is the power of **streaming graph execution output**.

Traditionally, when you execute a LangGraph agent using methods like `invoke()`, the entire computation runs to completion, and only then do you receive the final result. This is akin to ordering a multi-course meal and waiting for all dishes to be prepared and served simultaneously. While effective for batch processing or simple, quick tasks, it can lead to a poor user experience for long-running or interactive agents.

### Why Stream?

Streaming, on the other hand, allows you to receive incremental updates as your agent's graph executes. Think of it like watching a live stream of a cooking show: you see each ingredient being added, each step being performed, and the dish slowly coming together. In LangGraph, this means:

1.  **Enhanced User Experience**: Users get immediate feedback, reducing perceived latency and making the agent feel more interactive and intelligent.
2.  **Transparency and Debugging**: You can observe the agent's internal monologue, tool calls, and decision-making process as they happen. This is invaluable for understanding, debugging, and refining complex agentic workflows.
3.  **Real-time Interaction**: For conversational agents, streaming allows you to display LLM tokens as they are generated, creating a more natural and engaging chat experience.
4.  **Partial Results**: In scenarios where a full answer might take a long time, streaming can provide partial or intermediate results, keeping the user informed.

### How LangGraph Streams

LangGraph's `stream()` method provides an iterator that yields the state changes and outputs of each node as they complete. Instead of a single final output, you get a sequence of dictionaries, each representing a snapshot of the graph's state or the output of a specific node at a particular point in time. This allows you to process these updates incrementally, displaying them to the user or logging them for analysis.

This lesson will demonstrate how to leverage `stream()` to gain real-time insights into your agent's execution, making your AI applications more responsive and transparent.


In [ ]:
import operator
from typing import Annotated, Sequence, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

# --- 1. Define the Agent's State ---
# The state will track messages and any tool calls made.
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    tool_calls: list # To store any tool calls made

# --- 2. Define Tools ---
# Let's create a simple dummy tool for demonstration.
@tool
def search_web(query: str) -> str:
    """Searches the web for the given query and returns the results."""
    print(f"\n--- Executing Tool: search_web with query: '{query}' ---")
    # Simulate a delay and a result
    import time
    time.sleep(1.5)
    if "weather" in query.lower():
        return "The weather in London is 15°C and partly cloudy."
    elif "capital of france" in query.lower():
        return "The capital of France is Paris."
    else:
        return f"No specific answer found for '{query}'. Generic search result: Information about {query}."

# --- 3. Define Nodes (Agent Logic) ---

# A dummy LLM node that decides whether to use a tool or respond directly.
# In a real scenario, this would be an actual LLM call.
def call_llm(state: AgentState) -> AgentState:
    messages = state['messages']
    last_message = messages[-1]
    print(f"\n--- LLM Node: Processing message: '{last_message.content}' ---")

    # Simulate LLM decision logic
    if "weather" in last_message.content.lower() or "capital" in last_message.content.lower():
        # Simulate a tool call request from the LLM
        tool_call_message = AIMessage(
            content="",
            tool_calls=[
                {
                    "name": "search_web",
                    "args": {"query": last_message.content},
                    "id": "call_123"
                }
            ]
        )
        print("LLM decided to call a tool.")
        return {"messages": [tool_call_message], "tool_calls": tool_call_message.tool_calls}
    else:
        # Simulate a direct LLM response
        response_content = f"I received your query: '{last_message.content}'. I'm processing it directly. Here's a simulated response: This is a general answer to your question."
        print("LLM decided to respond directly.")
        return {"messages": [AIMessage(content=response_content)]}

# A node to execute the tool calls.
def call_tool(state: AgentState) -> AgentState:
    tool_calls = state['tool_calls']
    tool_results = []
    print(f"\n--- Tool Node: Executing {len(tool_calls)} tool calls ---")
    for tool_call in tool_calls:
        tool_name = tool_call['name']
        tool_args = tool_call['args']
        if tool_name == "search_web":
            result = search_web(**tool_args)
            tool_results.append(HumanMessage(content=result, name=tool_name, tool_call_id=tool_call['id']))
    print("Tool execution complete.")
    return {"messages": tool_results}

# --- 4. Build the Graph ---
workflow = StateGraph(AgentState)

workflow.add_node("llm", call_llm)
workflow.add_node("tool_executor", call_tool)

workflow.set_entry_point("llm")

# Conditional edge: if the LLM suggests tool calls, go to tool_executor, otherwise end.
def route_agent(state: AgentState):
    last_message = state['messages'][-1]
    if last_message.tool_calls:
        print("Routing to tool_executor.")
        return "tool_executor"
    else:
        print("Routing to END (direct LLM response).")
        return END

workflow.add_conditional_edges(
    "llm", # From the LLM node
    route_agent, # Use this function to determine the next node
    {
        "tool_executor": "tool_executor",
        END: END
    }
)

# After tool execution, the agent should always end (for this simple example).
workflow.add_edge("tool_executor", END)

# Compile the graph
app = workflow.compile()

# --- 5. Demonstrate Streaming vs. Non-Streaming ---

print("\n==================================================")
print("DEMONSTRATION 1: Non-Streaming (invoke())")
print("==================================================")

# Example 1: Non-streaming execution
# You'll see all print statements and then the final output at once.
print("\nInvoking agent with 'What is the weather in London?' (non-streaming)...\n")
final_state_invoke = app.invoke({"messages": [HumanMessage(content="What is the weather in London?")]})
print("\n--- Final State (invoke): ---")
print(final_state_invoke['messages'][-1].content)

print("\n==================================================")
print("DEMONSTRATION 2: Streaming (stream())")
print("==================================================")

# Example 2: Streaming execution
# You'll see output chunks as they are generated by each node.
print("\nStreaming agent with 'What is the capital of France?'...\n")

# The stream() method yields dictionaries representing state changes at each step.
for s in app.stream({"messages": [HumanMessage(content="What is the capital of France?")]}):
    # 's' is a dictionary where keys are node names and values are the output of that node.
    # The '__end__' key indicates the final state.
    if "__end__" not in s:
        for key, value in s.items():
            print(f"Node '{key}' outputted: {value}")
    else:
        print(f"Graph finished. Final state: {s['__end__']}")

print("\nStreaming agent with 'Tell me a joke.' (direct LLM response)...\n")
for s in app.stream({"messages": [HumanMessage(content="Tell me a joke.")]}):
    if "__end__" not in s:
        for key, value in s.items():
            print(f"Node '{key}' outputted: {value}")
    else:
        print(f"Graph finished. Final state: {s['__end__']}")


### Interpreting Streamed Output and Use Cases

When you run the `stream()` method, you'll notice that the output is not a single, monolithic result, but rather a sequence of dictionaries. Each dictionary represents a step in the graph's execution, typically corresponding to a node completing its work or the graph reaching its final state.

In our example, for the query "What is the capital of France?", you would observe:

1.  **`Node 'llm' outputted: {'messages': [AIMessage(content='', tool_calls=[{'name': 'search_web', 'args': {'query': 'What is the capital of France?'}, 'id': 'call_123'}])], 'tool_calls': [{'name': 'search_web', 'args': {'query': 'What is the capital of France?'}, 'id': 'call_123'}]}`**: This indicates that the `llm` node has processed the input and decided to make a tool call. The `messages` list now contains an `AIMessage` with `tool_calls` information, and the `tool_calls` state is updated.
2.  **`Node 'tool_executor' outputted: {'messages': [HumanMessage(content='The capital of France is Paris.', name='search_web', tool_call_id='call_123')]}`**: This shows that the `tool_executor` node has successfully run the `search_web` tool, and its output (the tool's result) is added to the `messages` list as a `HumanMessage`.
3.  **`Graph finished. Final state: {'messages': [HumanMessage(content='What is the capital of France?'), AIMessage(content='', tool_calls=[{'name': 'search_web', 'args': {'query': 'What is the capital of France?'}, 'id': 'call_123'}]), HumanMessage(content='The capital of France is Paris.', name='search_web', tool_call_id='call_123')], 'tool_calls': [{'name': 'search_web', 'args': {'query': 'What is the capital of France?'}, 'id': 'call_123'}]}`**: Finally, the `__end__` key signifies that the graph has completed its execution, and the value associated with it is the final state of the graph.

For the query "Tell me a joke.", you would see:

1.  **`Node 'llm' outputted: {'messages': [AIMessage(content="I received your query: 'Tell me a joke.'. I'm processing it directly. Here's a simulated response: This is a general answer to your question.")]}`**: The `llm` node processed the input and directly generated a response, as no tool call was deemed necessary.
2.  **`Graph finished. Final state: {'messages': [HumanMessage(content='Tell me a joke.'), AIMessage(content="I received your query: 'Tell me a joke.'. I'm processing it directly. Here's a simulated response: This is a general answer to your question.")], 'tool_calls': []}`**: The graph ends, providing the final state with the direct LLM response.

Notice how the `invoke()` method only gives you the final state, while `stream()` provides a step-by-step breakdown, including the intermediate outputs of each node.

### Performance Trade-offs and Typical Use Cases

**Performance:**

*   **Latency Perception**: Streaming significantly reduces the *perceived* latency for users, as they don't have to wait for the entire process to complete before seeing any output. This is a huge win for user experience.
*   **Actual Latency**: While streaming improves perceived latency, the total execution time of the graph might be marginally higher due to the overhead of yielding and processing multiple smaller chunks of data. However, this overhead is usually negligible compared to the benefits for interactive applications.
*   **Resource Usage**: Streaming can sometimes involve more frequent state updates and data transfers, potentially increasing CPU or memory usage slightly, especially if you're processing and storing every intermediate state. For most applications, this is not a bottleneck.

**Typical Use Cases:**

*   **Interactive Chatbots and Conversational AI**: Displaying LLM tokens as they are generated, showing tool calls, and indicating when the agent is thinking or performing an action.
*   **Complex Agentic Workflows**: Providing real-time feedback on the agent's progress through multi-step tasks, such as data analysis pipelines, research agents, or automation workflows.
*   **Debugging and Monitoring**: Developers can use streaming to get a granular view of how their agent is behaving, making it easier to identify issues or unexpected behavior.
*   **Building Responsive UIs**: Front-end applications can consume the streamed output and update the UI dynamically, creating a much smoother and more engaging user experience.
*   **Long-Running Tasks**: For agents that might take several seconds or even minutes to complete, streaming ensures the user is never left wondering if the agent is still working.

By embracing streaming, you empower your LangGraph agents to be more transparent, responsive, and user-friendly, which is crucial for building world-class AI applications in 2026 and beyond.


### Resources

*   **LangGraph Documentation on Streaming**: [https://langchain-ai.github.io/langgraph/how-to/stream/](https://langchain-ai.github.io/langgraph/how-to/stream/)
*   **LangChain Expression Language (LCEL) Streaming**: [https://python.langchain.com/docs/expression_language/streaming/](https://python.langchain.com/docs/expression_language/streaming/) (LangGraph builds upon LCEL's streaming capabilities)
*   **LangGraph GitHub Repository**: [https://github.com/langchain-ai/langgraph](https://github.com/langchain-ai/langgraph)
*   **LangChain Blog - Building Agentic RAG with LangGraph**: [https://blog.langchain.dev/agentic-rag-with-langgraph/](https://blog.langchain.dev/agentic-rag-with-langgraph/) (Often features streaming in examples)
